In [ ]:
import sys 
import os
from datetime import datetime
today = datetime.now().strftime("%Y-%m-%d")

import anndata as ad
import hdf5plugin
import squidpy as sq
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

from spatialdata_io.readers.xenium import xenium_aligned_image
import spatialdata as sd
import rapids_singlecell as rsc

import warnings
warnings.filterwarnings('ignore', category=ad.ImplicitModificationWarning)

# Add repo path to sys path (allows to access scripts and metadata from repo)
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses'
sys.path.insert(1, repo_path) 
sys.path.insert(2, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/scripts')
sys.path.insert(3, '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts')

# Autoreload custom scripts
%load_ext autoreload
%autoreload 2

# Define paths
plots_path = f'{repo_path}/plots/'
data_path = f'{repo_path}/data/'
model_path = os.path.join(repo_path, 'models')
general_data_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/data/'

print('Dir for plots: {}'.format(plots_path))
print('Dir for data: {}'.format(data_path))

# Formatting
from matplotlib import font_manager
font_manager.fontManager.addfont("/nfs/team205/ny1/ThymusSpatialAtlas/software/Arial.ttf")
plt.style.use('/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/General_analysis/scripts/plotting/thyAgeing.mplstyle')

from plotting.utils import calc_figsize, thyAgeing_greys, thyAgeing_colors

In [ ]:
from utils import get_latest_version,update_obs

xenium_adata = ad.read_h5ad(f'{data_path}/morphoFM/thyAging_morphoFM_mpp025_mf2.h5ad')
# Update metadata
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)

# Add sex information
xenium_adata.obs = xenium_adata.obs.reset_index(names = 'tile_id').merge(latest_meta[['donor', 'sex', 'age_group']].drop_duplicates(), left_on = 'DonorID', right_on = 'donor', how = 'left').set_index('tile_id')
xenium_adata.obs['tissue_embedding'] = 'FFPE'
xenium_adata.obs['is_xenium'] = True

In [ ]:
# Update metadata
from utils import get_latest_version,update_obs
latest_meta_path = get_latest_version(dir = f'{general_data_path}/metadata', file_prefix='Thymus_ageing_metadata')
latest_meta = pd.read_excel(latest_meta_path)

In [ ]:
# Retruieve new 
def _walk(path, depth):
    """Recursively list files and directories up to a certain depth"""
    depth -= 1
    with os.scandir(path) as p:
        for entry in p:
            yield entry.path
            if entry.is_dir() and depth > 0:
                yield from _walk(entry.path, depth)

solo_he_files = [f for f in _walk('/lustre/scratch126/cellgen/teichmann/lm25/data/thymus_he/thyAgeing_HnE', 2) if f.endswith('all_parts_mpp025_uni2h_morpho.h5ad')]
len(solo_he_files)

In [ ]:
# Load H&E data and add metadata
he_meta = pd.read_csv('/lustre/scratch126/cellgen/teichmann/lm25/data/thymus_he/thyAgeing_HnE.csv')
he_meta = he_meta.merge(latest_meta[['donor', 'sex', 'age_group']].drop_duplicates(), left_on = 'DonorID', right_on = 'donor', how = 'left').drop('donor', axis = 1)
he_meta['SampleID'] = he_meta['file_name'].str.split(' - ').str[0]
he_meta

In [ ]:
from scipy.sparse import csr_matrix

he_adata = []
for file in solo_he_files:
    sample_name = os.path.basename(file).split('_THY_tile_all_parts')[0]
    if sample_name not in he_meta['SampleID'].values:
        print(f'Sample {sample_name} not found in metadata, skipping...')
        continue

In [ ]:
from scipy.sparse import csr_matrix

he_adata = []
for file in solo_he_files:
    sample_name = os.path.basename(file).split('_THY_tile_all_parts')[0]
    sample_row = he_meta[he_meta['SampleID'] == sample_name]
    donor, age, sex, has_xenium, tissue_embedding, source = sample_row[['DonorID', 'age_group', 'sex', 'has_xenium', 'tissue_embedding', 'source']].values[0]
    ad_raw = ad.read_h5ad(file)
    _adata = ad.AnnData(X=csr_matrix(ad_raw.obsm['mf2_tile_uni2h_gray']), obs=ad_raw.obs)
    _adata.obs.drop(columns=['has_xenium'], inplace=True, errors='ignore')
    _adata.obs[['SampleID', 'DonorID', 'sex', 'age_group', 'tissue_embedding', 'has_xenium', 'source']] = [sample_name, donor, sex, age, tissue_embedding, has_xenium, source]
    he_adata.append(_adata)
    
he_adata = ad.concat(he_adata, axis=0)
#he_adata.obs.drop(columns=['has_xenium'], inplace=True, errors='ignore')
#he_adata.obs = he_adata.obs.reset_index(names = 'tile_id').merge(he_meta[['SampleID', 'DonorID', 'sex', 'age_group', 'tissue_embedding', 'has_xenium']].drop_duplicates(), left_on = 'SampleID', right_on = 'SampleID', how = 'left').set_index('tile_id')

he_adata

In [ ]:
he_adata.obs[['SampleID', 'DonorID', 'age_group', 'sex', 'has_xenium', 'tissue_embedding', 'source']].drop_duplicates()

In [ ]:
# Remove OCT samples
he_adata = he_adata[(~he_adata.obs['tissue_embedding'].str.contains('OCT')) & (~he_adata.obs['source'].str.contains('NIH')), :].copy()

# L2 normalise per patch: maintains how UNI2 represents a patch -> features are supposed tp be compared by cosine similarity
from sklearn.preprocessing import normalize
he_adata.X = csr_matrix(normalize(he_adata.X, norm='l2', axis=1)).copy()

In [ ]:
# Perform PCA (with L2 norm)
rsc.get.anndata_to_GPU(he_adata)
rsc.pp.pca(he_adata, n_comps = 50)

with plt.rc_context({'figure.figsize' : calc_figsize(width = 270, height = 180)}):
    sc.pl.pca(he_adata, color = ['DonorID', 'age_group', 'sex', 'tissue_embedding', 'has_xenium', 'source'], legend_fontsize = 10, ncols = 2, wspace=0.3, return_fig = True, show = False)
    plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_he_pca.png', bbox_inches='tight', dpi = 300)

In [ ]:
# Extract variance explained by each principal component
pca_variance = he_adata.uns['pca']['variance']
pca_variance_ratio = he_adata.uns['pca']['variance_ratio']

# Create a dataframe with variance information
variance_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(pca_variance))],
    'Variance': pca_variance,
    'Variance_Ratio': pca_variance_ratio,
    'Cumulative_Variance_Ratio': np.cumsum(pca_variance_ratio)
})

print(variance_df.iloc[:10])  # Display the first 10 rows of the variance dataframe

In [ ]:
from scipy import stats

# Test whether PC1 separates he_adata.obs['has_xenium'] using statistical test

# Get PC1 values for each group
pc1_values = he_adata.obsm['X_pca'][:, 0]
group_xenium = he_adata.obs['has_xenium']

# Separate PC1 values by group
pc1_xenium_true = pc1_values[group_xenium == True]
pc1_xenium_false = pc1_values[group_xenium == False]

# Perform t-test
t_stat, p_value = stats.ttest_ind(pc1_xenium_true, pc1_xenium_false)

print(f"PC1 separation by has_xenium:")
print(f"Mean PC1 (has_xenium=True): {pc1_xenium_true.mean():.4f}")
print(f"Mean PC1 (has_xenium=False): {pc1_xenium_false.mean():.4f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.2e}")
print()

# Create a boxplot to visualize separation
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
data_to_plot = [pc1_xenium_false, pc1_xenium_true]
bp = ax.boxplot(data_to_plot, labels=['has_xenium=False', 'has_xenium=True'], patch_artist=True)
ax.set_ylabel('PC1')
ax.set_title('PC1 separation by has_xenium')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Integrate batches with harmony, using multiple batch keys
batch_cols = ['DonorID', 'sex', 'has_xenium']
rsc.pp.harmony_integrate(he_adata, key = batch_cols)
rsc.pp.neighbors(he_adata, use_rep = 'X_pca_harmony', n_neighbors = 30)
rsc.tl.umap(he_adata, min_dist = 0.3)

with plt.rc_context({'figure.figsize' : calc_figsize(width = 270, height = 180)}):
    sc.pl.umap(he_adata, color = ['DonorID', 'age_group', 'sex', 'tissue_embedding', 'has_xenium', 'source'], legend_fontsize = 10, ncols = 2, wspace=0.3, return_fig = True, show = False)
    plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_he_harmony_umap.png', bbox_inches='tight', dpi = 300)

Still strong separation along technical axis (has_xenium) -> check difference in embedding space for donors with and without Xenium data

In [ ]:
# ---- Size-independent integration metrics for morphoFM embeddings ----
# t-stat scales with sqrt(N) and is meaningless at 1e5-1e6 tiles. These are not.
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler


def cohens_d(x0, x1):
    """Standardized mean difference — effect size, independent of N."""
    n0, n1 = len(x0), len(x1)
    s = np.sqrt(((n0 - 1) * x0.var(ddof=1) + (n1 - 1) * x1.var(ddof=1)) / (n0 + n1 - 2))
    return (x1.mean() - x0.mean()) / s if s > 0 else 0.0


def eta_squared(x, groups):
    """Fraction of a variable's variance explained by group label (0..1)."""
    grand = x.mean()
    ss_tot = ((x - grand) ** 2).sum()
    ss_bet = sum(len(x[groups == g]) * (x[groups == g].mean() - grand) ** 2
                 for g in np.unique(groups))
    return ss_bet / ss_tot if ss_tot > 0 else 0.0


def ilisi(rep, batch, k=30, subsample=50000, seed=0):
    """
    Mean inverse-Simpson index of the batch label over each cell's kNN.
    2 batches -> range [1, 2]; ~2 = well mixed, ~1 = fully separated.
    """
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(rep), min(subsample, len(rep)), replace=False)
    R, b = rep[idx], np.asarray(batch)[idx]
    nn = NearestNeighbors(n_neighbors=k + 1).fit(R)
    neigh = nn.kneighbors(R, return_distance=False)[:, 1:]  # drop self
    labs = b[neigh]
    vals = np.unique(b)
    p = np.stack([(labs == v).mean(axis=1) for v in vals], axis=1)  # (cells, nbatch)
    return float(np.mean(1.0 / np.sum(p ** 2, axis=1)))


def evaluate_representation(adata, rep_key, batch_key='has_xenium',
                            subsample=100000, seed=0):
    """Cohen's d (LDA axis), platform variance share, classifier AUC, iLISI."""
    X = np.asarray(adata.obsm[rep_key])
    b = adata.obs[batch_key].astype(int).values

    # 1) Best-case separability effect size: Cohen's d on the LDA discriminant axis.
    z = LinearDiscriminantAnalysis(n_components=1).fit_transform(X, b).ravel()
    d = abs(cohens_d(z[b == 0], z[b == 1]))

    # 2) Share of embedding variance attributable to platform (eta^2 per dim,
    #    weighted by each dim's variance).
    var = X.var(axis=0)
    eta = np.array([eta_squared(X[:, j], b) for j in range(X.shape[1])])
    platform_var_share = float(np.sum(var * eta) / np.sum(var))

    # 3) Can a linear classifier still tell platforms apart? AUC 0.5=mixed, 1=separable.
    rng = np.random.default_rng(seed)
    sub = rng.choice(len(X), min(subsample, len(X)), replace=False)
    Xs = StandardScaler().fit_transform(X[sub])
    auc = cross_val_score(LogisticRegression(max_iter=1000), Xs, b[sub],
                          cv=StratifiedKFold(5, shuffle=True, random_state=seed),
                          scoring='roc_auc').mean()

    # 4) Local mixing.
    il = ilisi(X, b, subsample=min(subsample, 50000), seed=seed)

    return {'rep': rep_key, 'cohens_d': round(d, 3),
            'platform_var_share': round(platform_var_share, 4),
            'classifier_auc': round(float(auc), 3), 'iLISI': round(il, 3)}


In [ ]:
import pandas as pd

# Representations to compare: raw PCA vs Harmony output (add whatever you've run).
reps = [r for r in ['X_pca', 'X_pca_harmony'] if r in he_adata.obsm]
summary = pd.DataFrame([evaluate_representation(he_adata, r) for r in reps])
print("=== Global metrics (lower cohens_d / platform_var_share / AUC, "
      "higher iLISI = better mixed) ===")
print(summary.to_string(index=False))

# ---- Paired-donor analysis: the ONLY biology-controlled read on the technical effect ----
# For A67/A70/A72 the two platforms are the SAME tissue, so their embedding gap is
# pure platform effect. Compare it to the typical gap between DIFFERENT donors on the
# SAME platform (a biological yardstick). If platform gap << donor gap, stain norm did
# its job; if comparable/larger, a real technical residual remains.
paired = ['A67', 'A70', 'A72', 'U48', 'A45']

def centroid(mask, rep='X_pca'):
    return np.asarray(he_adata.obsm[rep])[mask.values].mean(axis=0)

for rep in reps:
    print(f"\n=== Paired-donor distances in {rep} ===")
    obs = he_adata.obs
    plat_gaps = []
    for d in paired:
        mx = (obs['DonorID'] == d) & (obs['has_xenium'] == True)
        mn = (obs['DonorID'] == d) & (obs['has_xenium'] == False)
        if mx.sum() and mn.sum():
            gap = np.linalg.norm(centroid(mx, rep) - centroid(mn, rep))
            plat_gaps.append(gap)
            print(f"  {d}: |Xenium - nonXenium| = {gap:.3f}  "
                  f"(n_xen={int(mx.sum())}, n_non={int(mn.sum())})")

    # Biological yardstick: pairwise centroid distances between non-Xenium donors.
    non_donors = obs.loc[obs['has_xenium'] == False, 'DonorID'].unique()
    cents = {d: centroid((obs['DonorID'] == d) & (obs['has_xenium'] == False), rep)
             for d in non_donors}
    bio_gaps = [np.linalg.norm(cents[a] - cents[b])
                for i, a in enumerate(non_donors) for b in non_donors[i + 1:]]
    print(f"  --> mean PLATFORM gap (paired):        {np.mean(plat_gaps):.3f}")
    print(f"  --> mean BIOLOGICAL gap (donor-donor): {np.mean(bio_gaps):.3f}")
    print(f"  --> platform/biology ratio: {np.mean(plat_gaps)/np.mean(bio_gaps):.2f}"
          "   (<<1 good, ~1+ means technical effect rivals biology)")

# Bonus: iLISI restricted to the 3 paired donors — where platforms SHOULD overlap.
mask_paired = he_adata.obs['DonorID'].isin(paired).values
for rep in reps:
    il_p = ilisi(np.asarray(he_adata.obsm[rep])[mask_paired],
                 he_adata.obs['has_xenium'][mask_paired].astype(int).values)
    print(f"iLISI on paired donors only [{rep}]: {il_p:.3f}  (2.0 = perfect mixing)")

1. Harmony is doing essentially nothing (AUC 1.0 → 1.0, iLISI 1.000 → 1.003, platform_var_share 0.255 → 0.251). That's not a tuning failure — it's structural. AUC = 1.0 and iLISI = 1.0 mean the two platforms share zero k-NN neighborhoods; they're perfectly linearly separable. Harmony works by finding clusters shared across batches and aligning their centroids — when there are no shared neighborhoods, it has nothing to anchor on. Stop tuning Harmony; it cannot fix a fully-separable batch effect.

2. The residual is real, technical, and large. The paired-donor ratio of 1.69 is the key result: with biology held fixed (same donor, same tissue), the platform gap (0.62) is ~1.7× the typical between-donor biological gap (0.37). So this is not confounded biology masquerading as batch — it's a genuine technical effect that's bigger than the biological signal you care about. Correction is justified (this is the "≈1+" case I flagged, not the "<<1, don't force it" case).

3. The effect looks low-dimensional and roughly linear — 25% of variance concentrated along essentially one LDA axis (Cohen's d ~13). That's exactly the regime where a paired-anchor linear correction works well. This is the moment your 3 paired donors pay off.

In [ ]:
import numpy as np
obs = he_adata.obs
X = np.asarray(he_adata.obsm['X_pca'])

# Per-donor shift vectors (map Xenium -> non-Xenium reference), biology held fixed.
shifts = {}
for d in paired:
    mx = ((obs['DonorID'] == d) & (obs['has_xenium'] == True)).values
    mn = ((obs['DonorID'] == d) & (obs['has_xenium'] == False)).values
    print(f"{d}: n_xenium={int(mx.sum())}, n_nonxenium={int(mn.sum())}")
    if mx.sum() and mn.sum():
        shifts[d] = X[mn].mean(0) - X[mx].mean(0)

ds = list(shifts)

# (a) Is it one global direction? Cosine ~1 = yes, a single subtraction is valid.
print("\nShift-vector consistency (want cos ~1):")
for i, a in enumerate(ds):
    for b in ds[i + 1:]:
        cos = shifts[a] @ shifts[b] / (np.linalg.norm(shifts[a]) * np.linalg.norm(shifts[b]))
        print(f"  {a} vs {b}: cos={cos:.3f}  |{a}|={np.linalg.norm(shifts[a]):.3f}  "
              f"|{b}|={np.linalg.norm(shifts[b]):.3f}")

# (b) Leave-one-donor-out: does a shift learned on OTHER donors fix the held-out one?
#     This is the real test that a global correction generalizes (vs overfitting).
print("\nLeave-one-donor-out validation (biological gap ~0.37):")
for test in ds:
    train = [d for d in ds if d != test]
    sh = np.mean([shifts[d] for d in train], axis=0)
    mx = ((obs['DonorID'] == test) & (obs['has_xenium'] == True)).values
    mn = ((obs['DonorID'] == test) & (obs['has_xenium'] == False)).values
    before = np.linalg.norm(X[mx].mean(0) - X[mn].mean(0))
    after = np.linalg.norm((X[mx].mean(0) + sh) - X[mn].mean(0))
    print(f"  {test}: gap {before:.3f} -> {after:.3f} using shift from {train}")

# (c) Apply the averaged shift to ALL Xenium tiles and re-score.
shift = np.mean([shifts[d] for d in ds], axis=0)
Xc = X.copy()
Xc[(obs['has_xenium'] == True).values] += shift
he_adata.obsm['X_pca_platcorr'] = Xc

print("\n=== After paired-anchor correction ===")
print(evaluate_representation(he_adata, 'X_pca_platcorr'))
mp = obs['DonorID'].isin(paired).values
print("iLISI paired-only:",
      round(ilisi(Xc[mp], obs['has_xenium'][mp].astype(int).values), 3))

# Then, to see it: sc.pp.neighbors(he_adata, use_rep='X_pca_platcorr', n_neighbors=30)
#                   sc.tl.umap(he_adata, min_dist=0.3)


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

obs = he_adata.obs

def paired_auc(rep):
    """Within-paired-donor platform separability — the confound-free residual metric."""
    m = obs['DonorID'].isin(paired).values
    X = StandardScaler().fit_transform(np.asarray(he_adata.obsm[rep])[m])
    y = obs['has_xenium'][m].astype(int).values
    overall = cross_val_score(LogisticRegression(max_iter=1000), X, y,
                              cv=StratifiedKFold(5, shuffle=True, random_state=0),
                              scoring='roc_auc').mean()
    print(f"[{rep}] within-paired-donor AUC (0.5=technically mixed): {overall:.3f}")

for rep in ['X_pca', 'X_pca_platcorr']:
    paired_auc(rep)

Within-paired-donor residual just with mean shift still too high -> apply anchor to location and scale:

In [ ]:
from scipy.sparse import issparse
from sklearn.decomposition import PCA

rsc.get.anndata_to_CPU(he_adata)
is_xen = (obs['has_xenium'] == True).values
Xf = he_adata.X                                   # L2-normed 1536-d features
X = (Xf.toarray() if issparse(Xf) else np.asarray(Xf)).astype(np.float32)

# Estimate location + scale from paired donors ONLY (biology-matched -> not composition).
xp = (obs['DonorID'].isin(paired) & is_xen).values
npd = (obs['DonorID'].isin(paired) & ~is_xen).values
mu_x, mu_n = X[xp].mean(0), X[npd].mean(0)
scale = np.clip((X[npd].std(0) + 1e-6) / (X[xp].std(0) + 1e-6), 0.5, 2.0)

# Match all Xenium tiles to the non-Xenium location & scale, then re-PCA.
X[is_xen] = (X[is_xen] - mu_x) * scale + mu_n
he_adata.obsm['X_pca_ls'] = PCA(n_components=50, random_state=0).fit_transform(X)

print(evaluate_representation(he_adata, 'X_pca_ls'))
paired_auc('X_pca_ls')

Platforms are now mixed within donors. Check whether biological variation is still preserved (donor- and age-level):

In [ ]:
def var_share(rep, key, mask=None):
    X = np.asarray(he_adata.obsm[rep])
    g = he_adata.obs[key].values
    if mask is not None:
        X, g = X[mask], g[mask]
    var = X.var(0)
    eta = np.array([eta_squared(X[:, j], g) for j in range(X.shape[1])])
    return float((var * eta).sum() / var.sum())

non = (~he_adata.obs['has_xenium'].astype(bool)).values  # platform held constant
print(f"{'rep':16s} platform   age(nonXen)  donor(nonXen)")
for rep in ['X_pca', 'X_pca_harmony', 'X_pca_ls']:
    if rep in he_adata.obsm:
        print(f"{rep:16s} {var_share(rep,'has_xenium'):.4f}     "
              f"{var_share(rep,'age_group',non):.4f}       "
              f"{var_share(rep,'DonorID',non):.4f}")

Platform variation is reduced while age and donor-level variation are preserved, so using this embedding for clustering!

In [ ]:
# Record scaling
mu_x, mu_n, scale

In [ ]:
import gc
import cupy as cp

# Integrate batches with harmony, using multiple batch keys
# Free GPU/CUDA memory from previous RAPIDS/CuPy steps
rsc.get.anndata_to_CPU(he_adata)
gc.collect()
try:
    cp.cuda.runtime.deviceSynchronize()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()
except Exception as e:
    print(f"CUDA cleanup skipped: {e}")
    
rsc.get.anndata_to_GPU(he_adata)
rsc.pp.neighbors(he_adata, use_rep = 'X_pca_ls', n_neighbors = 30)
rsc.tl.umap(he_adata, min_dist = 0.3)

with plt.rc_context({'figure.figsize' : calc_figsize(width = 270, height = 180)}):
    sc.pl.umap(he_adata, color = ['DonorID', 'age_group', 'sex', 'tissue_embedding', 'has_xenium', 'source'], legend_fontsize = 10, ncols = 2, wspace=0.3, return_fig = True, show = False)
    plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_he_pca_ls_umap.png', bbox_inches='tight', dpi = 300)

In [ ]:
import matplotlib.pyplot as plt, numpy as np
paired = ['A67', 'A70', 'A72', 'U48']
obs, um = he_adata.obs, he_adata.obsm['X_umap']
mask = obs['DonorID'].isin(paired).values
xen = obs['has_xenium'][mask].astype(bool).values
sub = um[mask]

fig, ax = plt.subplots(1, 3, figsize=(18, 5))
# (1) paired donors highlighted on the global UMAP, split by platform
ax[0].scatter(um[:, 0], um[:, 1], s=1, c='lightgrey')
ax[0].scatter(sub[~xen, 0], sub[~xen, 1], s=4, c='tab:blue', label='paired non-Xen')
ax[0].scatter(sub[xen, 0], sub[xen, 1], s=4, c='tab:red', label='paired Xen')
ax[0].legend(); ax[0].set_title('Paired donors (matched biology)')
# (2) global UMAP by donor  (3) by age -> is the split really donor/age?
for a, key in zip(ax[1:], ['DonorID', 'age_group']):
    for v in obs[key].unique():
        m = (obs[key] == v).values
        a.scatter(um[m, 0], um[m, 1], s=1, label=str(v))
    a.set_title(f'global UMAP by {key}')
plt.show()

# quantify: local mixing WITHIN paired donors on the corrected rep (nonlinear/local)
print("paired-only iLISI [X_pca_ls]:",
      round(ilisi(np.asarray(he_adata.obsm['X_pca_ls'])[mask], xen.astype(int)), 3))


In [ ]:
def paired_ilisi_balanced(rep, k=30, seed=0):
    m = obs['DonorID'].isin(paired).values
    X = np.asarray(he_adata.obsm[rep])[m]; y = obs['has_xenium'][m].astype(int).values
    rng = np.random.default_rng(seed)
    n = min((y == 0).sum(), (y == 1).sum())
    idx = np.concatenate([rng.choice(np.where(y == 0)[0], n, False),
                          rng.choice(np.where(y == 1)[0], n, False)])
    return round(ilisi(X[idx], y[idx], k=k, subsample=len(idx)), 3)

print("balanced paired iLISI [X_pca_ls]:", paired_ilisi_balanced('X_pca_ls'))


Implement covariance matching (not just scale matching)

In [ ]:
import numpy as np
paired = ['A67', 'A70', 'A72', 'U48', 'A45']; obs = he_adata.obs
Z = np.asarray(he_adata.obsm['X_pca']).astype(np.float64)   # start from raw PCA; CORAL subsumes mean+scale
is_xen = (obs['has_xenium'] == True).values

def sym_pow(C, p, eps=1e-6):
    w, V = np.linalg.eigh(C); w = np.clip(w, eps, None)
    return (V * (w ** p)) @ V.T

xp = (obs['DonorID'].isin(paired) & is_xen).values
npd = (obs['DonorID'].isin(paired) & ~is_xen).values
mu_s, mu_t = Z[xp].mean(0), Z[npd].mean(0)
Cs = np.cov(Z[xp], rowvar=False) + 1e-4 * np.eye(Z.shape[1])
Ct = np.cov(Z[npd], rowvar=False) + 1e-4 * np.eye(Z.shape[1])
A = sym_pow(Cs, -0.5) @ sym_pow(Ct, 0.5)          # whiten by source, recolor to target

Z[is_xen] = (Z[is_xen] - mu_s) @ A + mu_t
he_adata.obsm['X_pca_coral'] = Z.astype(np.float32)

print(evaluate_representation(he_adata, 'X_pca_coral'))
paired_auc('X_pca_coral')
print("balanced paired iLISI [X_pca_coral]:", paired_ilisi_balanced('X_pca_coral'))


In [ ]:
with plt.rc_context({'figure.figsize' : calc_figsize(width = 270, height = 180)}):
    sc.pl.embedding(he_adata, basis = 'X_pca_coral', color = ['DonorID', 'age_group', 'sex', 'has_xenium', 'source'], legend_fontsize = 10, ncols = 2, wspace=0.3, return_fig = True, show = False)
    plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_he_pca_coral.png', bbox_inches='tight', dpi = 300)

In [ ]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

Z = np.asarray(he_adata.obsm['X_pca_coral']).astype(np.float32)  # build on the CORAL rep
obs = he_adata.obs
is_xen = (obs['has_xenium'] == True).values
paired = ['A67', 'A70', 'A72', 'U48', 'U45']; pm = obs['DonorID'].isin(paired).values

# 1) Mutual nearest neighbors between platforms WITHIN paired donors (matched biology).
xen_idx = np.where(pm & is_xen)[0]; non_idx = np.where(pm & ~is_xen)[0]
Zx, Zn = Z[xen_idx], Z[non_idx]
k = 15
_, xn = NearestNeighbors(n_neighbors=k).fit(Zn).kneighbors(Zx)   # xen -> non
_, nx = NearestNeighbors(n_neighbors=k).fit(Zx).kneighbors(Zn)   # non -> xen
nx_sets = [set(r) for r in nx]
anchors, corr = [], []
for i in range(len(xen_idx)):
    partners = [j for j in xn[i] if i in nx_sets[j]]
    if partners:
        anchors.append(i); corr.append(Zn[partners].mean(0) - Zx[i])  # xen->non displacement
anchors = np.array(anchors); corr = np.array(corr, np.float32)
print(f"MNN anchors: {len(anchors)} / {len(xen_idx)} paired-Xenium tiles")

# 2) Propagate the displacement field to ALL Xenium tiles (Gaussian-weighted from anchors).
nn_a = NearestNeighbors(n_neighbors=min(20, len(anchors))).fit(Zx[anchors])
allx = np.where(is_xen)[0]
d, a = nn_a.kneighbors(Z[allx])
w = np.exp(-(d ** 2) / (np.median(d) ** 2 + 1e-8)); w /= w.sum(1, keepdims=True)
Zc = Z.copy(); Zc[allx] += np.einsum('ij,ijk->ik', w, corr[a])
he_adata.obsm['X_pca_mnn'] = Zc

print(evaluate_representation(he_adata, 'X_pca_mnn'))
paired_auc('X_pca_mnn')
print("balanced paired iLISI [X_pca_mnn]:", paired_ilisi_balanced('X_pca_mnn'))

In [ ]:
he_adata.write_h5ad(f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad')

In [ ]:
!which python

In [ ]:
import anndata as ad
import os
repo_path,_ = os.path.split(os.path.split(os.getcwd())[0])
repo_path = '/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses'
data_path = f'{repo_path}/data/'

he_adata = ad.read_h5ad(f'{data_path}/morphoFM/thyAging_morphoFM_mpp025_mf2_he_coral.h5ad')

In [ ]:
# Batch-balanced kNN graph on the undistorted CORAL rep, then Leiden niches.
# (pip install bbknn if needed)
import scanpy as sc
rsc.pp.bbknn(he_adata, batch_key='has_xenium', use_rep='X_pca_coral',
                     neighbors_within_batch=3, n_pcs=50, algorithm = 'cagra')
rsc.tl.leiden(he_adata, resolution=1.0, key_added='leiden_r1.0')
rsc.tl.umap(he_adata, min_dist=0.3)

with plt.rc_context({'figure.figsize' : calc_figsize(width = 270, height = 180)}):
    sc.pl.umap(he_adata, color = ['leiden_r1.0', 'DonorID', 'age_group', 'has_xenium', 'source', 'sex'], legend_fontsize = 10, ncols = 2, wspace=0.3, return_fig = True, show = False)
    plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_bbknn_umap.png', bbox_inches='tight', dpi = 300)

In [ ]:
import pandas as pd, numpy as np
# (1) No niche should be platform-pure (that = a batch cluster, not a niche).
comp = pd.crosstab(he_adata.obs['leiden_r1.0'], he_adata.obs['has_xenium'], normalize='index')
print("Niches >95% one platform (suspect):",
      comp.index[(comp.max(1) > 0.95)].tolist())

# (2) Within paired donors, niche proportions must agree across platforms.
for d in ['A67', 'A70', 'A72', 'U48', 'A45']:
    s = he_adata.obs[he_adata.obs['DonorID'] == d]
    px = s[s.has_xenium.astype(bool)]['leiden_r1.0'].value_counts(normalize=True)
    pn = s[~s.has_xenium.astype(bool)]['leiden_r1.0'].value_counts(normalize=True)
    idx = px.index.union(pn.index)
    r = np.corrcoef(px.reindex(idx, fill_value=0), pn.reindex(idx, fill_value=0))[0, 1]
    print(f"{d}: niche-proportion corr (Xen vs non-Xen) = {r:.3f}")

In [ ]:
he_adata.write_h5ad(f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad')

## Map Xenium cell types to H&E patches

In [ ]:
he_adata = ad.read_h5ad(f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad')

In [ ]:
rsc.get.anndata_to_GPU(he_adata)
for res in [0.5, 1.0, 2.0, 3.0]:
    rsc.tl.leiden(he_adata, resolution=res, key_added=f'leiden_r{res}')
    
    
with plt.rc_context({'figure.figsize' : calc_figsize(width = 270, height = 180)}):
    sc.pl.umap(he_adata, color = [f'leiden_r{res}' for res in [0.5, 1.0, 2.0, 3.0]], legend_fontsize = 10, ncols = 2, wspace=0.3, return_fig = True, show = False)
    plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_bbknn_umap_clusters.png', bbox_inches='tight', dpi = 300)

In [ ]:
plt.show()

In [ ]:
import seaborn as sns
qc_cols = ['tissue_fraction', 'valid_tissue_fraction', 'saturation_fraction', 'dark_fraction', 'gray_std']

plt.figure(figsize=calc_figsize(width = 180, height = 270))
g = sns.FacetGrid(he_adata.obs.melt(id_vars='leiden_r2.0', value_vars=qc_cols), 
                    col='variable', col_wrap=1, sharey=False, height=4, aspect=2)
g.map_dataframe(sns.violinplot, x='leiden_r2.0', y='value', hue='leiden_r2.0', split=False, 
                palette=he_adata.uns['leiden_r2.0_colors'], inner=None)
g.add_legend()
plt.savefig(f'{plots_path}/morpho_niches2/thyAgeing_leiden_r2.0_qc_violin.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
he_adata.obs['leiden_r2.0'].value_counts()

In [ ]:
he_adata.obs.columns

In [ ]:
he_meta

In [ ]:
import tifffile
import numpy as np
import json
import numpy as np
import matplotlib.cm as cm

def tiles_to_original_coords(tiles_df, orig_mpp, target_mpp=0.25):
    """
    Map tile boxes (in the resampled mpp-image space) back to original
    level-0 WSI pixel coordinates.

    resampled = original * scale, with scale = orig_mpp / target_mpp
      => original = resampled / scale = resampled * (target_mpp / orig_mpp)
    """
    # Matches resample_to_target_mpp: if orig ~= target, no resampling was done.
    if abs(orig_mpp - target_mpp) <= 1e-4:
        inv_scale = 1.0
    else:
        scale = orig_mpp / target_mpp
        inv_scale = 1.0 / scale          # = target_mpp / orig_mpp

    out = tiles_df.copy()
    for col in ["x0", "x1", "y0", "y1", "cx_img", "cy_img"]:
        if col in out.columns:
            out[col + "_orig"] = out[col] * inv_scale

    return out, orig_mpp, inv_scale

from matplotlib.colors import to_rgb

def color_to_rgb255(c):
    """Any matplotlib color (hex, name, rgb tuple) -> [r, g, b] ints 0-255 for QuPath."""
    r, g, b = to_rgb(c)                       # floats 0..1
    return [int(round(r * 255)), int(round(g * 255)), int(round(b * 255))]


def export_niches_to_geojson(he_adata, sample, niche_col, orig_mpp, palette):
    """
    Export the patches of a given sample to a GeoJSON file with their niche classification.
    
    Parameters:
    - he_adata: AnnData object containing the data.
    - sample: Sample ID to filter the data.
    - niche_col: Column name in he_adata.obs that contains the niche classification.
    - orig_mpp: Original microns per pixel value.
    - palette: Dictionary mapping niche labels to colors.
    """

    obs = he_adata.obs[he_adata.obs['SampleID'] == sample].copy()
    obs = tiles_to_original_coords(obs, orig_mpp)[0]
    
    x0 = np.round(obs["x0"].to_numpy()).astype(int)
    x1 = np.round(obs["x1"].to_numpy()).astype(int)
    y0 = np.round(obs["y0"].to_numpy()).astype(int)
    y1 = np.round(obs["y1"].to_numpy()).astype(int)

    label_col = niche_col
    cats = list(obs[label_col].cat.categories)
    codes = obs[label_col].cat.codes.to_numpy()       # int per patch, -1 if NaN
    labels = obs[label_col].astype(str).to_numpy()

    features = []
    for i in range(len(obs)):
        a, b, c, d = float(x0[i]), float(x1[i]), float(y0[i]), float(y1[i])
        ring = [[a, c], [b, c], [b, d], [a, d], [a, c]]   # closed, CW
        features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [ring]},
            "properties": {
                "objectType": "annotation",
                "classification": {"name": labels[i], "color": color_to_rgb255(palette[labels[i]])},
            },
        })

    fc = {"type": "FeatureCollection", "features": features}
    with open(f"/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data/morpho_niches/{sample}_patches_qupath.geojson", "w") as f:
        json.dump(fc, f)
    print(len(features), "patches written")


In [ ]:
sample = 'A70.ome'
niche_col = 'leiden_r2.0'
orig_mpp = he_meta[he_meta['SampleID'] == sample]['resolution'].values[0]
palette = dict(zip(he_adata.obs[niche_col].cat.categories, he_adata.uns[f'{niche_col}_colors']))

export_niches_to_geojson(he_adata, sample, niche_col, orig_mpp, palette)

In [ ]:
# Transfer to CPU for saving and downstream analysis
rsc.get.anndata_to_CPU(he_adata)

In [ ]:
he_adata.write_h5ad(f'{data_path}/morphoFM/thyAging_CAM_morphoFM_mpp025_mf2_he_coral.h5ad')

In [ ]:
# Load xenium data for cell type reference and niche annotation. 
# The xenium data is already preprocessed and annotated with cell type predictions and CellCharter clusters.

xenium_adata = ad.read_h5ad(f'{data_path}/xenium/adata_xenium_2025-01-14.zarr')

sc.pp.filter_cells(xenium_adata, min_counts = 100)
knn_pred_df = pd.read_csv(f'{data_path}/xenium/ThyAge_xenium_scarches_annos_2025-04-09.csv', index_col = 0)
knn_cols = knn_pred_df.columns.tolist()
xenium_adata.obs[knn_cols] = knn_pred_df.reindex(xenium_adata.obs_names)

clusters = ['cluster_cellcharter', 'cluster_cellcharter6', 'cluster_cellcharter10']
xenium_adata.obs[clusters] = pd.read_csv(f'{data_path}/xenium/ThyAge_xenium_adata_paed+adult_CellCharter_clusters.csv', index_col = 0).reindex(xenium_adata.obs_names)

xenium_adata.obs['cluster_cellcharter6'] = xenium_adata.obs['cluster_cellcharter6'].astype("category")
cluster6anno  = {0: 'Cortex', 1: 'InflammatoryPVS', 2: 'Medulla', 3: 'Capsule/PVS', 4: 'Adipose', 5:'mcTEC-rich'}
cluster6order = ['Capsule/PVS', 'Cortex', 'Medulla', 'mcTEC-rich', 'InflammatoryPVS', 'Adipose']

xenium_adata.obs['niche_cellcharter6'] = xenium_adata.obs['cluster_cellcharter6'].map(cluster6anno)
xenium_adata.obs['niche_cellcharter6'] = xenium_adata.obs['niche_cellcharter6'].cat.reorder_categories(cluster6order).copy()


xenium_adata.obs['cluster_cellcharter'] = xenium_adata.obs['cluster_cellcharter'].astype("category")
cluster_anno  = {0: 'Cortex', 1: 'Medulla', 2: 'PVS'}
cluster3order = ['Cortex', 'Medulla', 'PVS']

xenium_adata.obs['niche_cellcharter3'] = xenium_adata.obs['cluster_cellcharter'].map(cluster_anno)
xenium_adata.obs['niche_cellcharter3'] = xenium_adata.obs['niche_cellcharter3'].cat.reorder_categories(cluster3order).copy()

xenium_anno = ad.read_h5ad('/nfs/team205/vk8/projects/thymus_ageing_atlas/Spatial_analyses/data/thyAgeing_morpho_xen_anno_2026-05-16.zarr', backed = 'r').obs[['taa_l3', 'taa_l2_v2']]
xenium_adata.obs = xenium_adata.obs.join(xenium_anno, how = 'left')

xenium_adata

In [ ]:
mpp = 0.25  
patch_px = 224
patch_um = patch_px * mpp  

he_adata.obs[['x0_um', 'y0_um', 'x1_um', 'y1_um']] = np.int16(he_adata.obs[['x0', 'y0', 'x1', 'y1']] * mpp)
he_adata.obs[['patch_col', 'patch_row']] = (he_adata.obs[['x0', 'y0']] // patch_px).astype(int) 
he_adata.obs.head()

In [ ]:
import numpy as np

alignment_dir = f'{data_path}/xenium/he_alignments'
PIXEL_SIZE    = 0.2125   # µm/px of the Xenium morphology image (matrix.csv domain)
HE_PIXEL_SIZE = 0.221    # µm/px of the H&E image
scale = HE_PIXEL_SIZE / mpp   # H&E native px -> target-mpp (0.25) px

def apply_affine(M, xy):
    """Apply a 3x3 affine to an (n,2) array of (x, y) points."""
    h = np.hstack([xy, np.ones((len(xy), 1))])
    return (h @ M.T)[:, :2]

xenium_coords_transformed = np.array([]).reshape(0, 2)  # Initialize empty array to hold transformed coordinates
xenium_obs_names = []
for donor in ['A67', 'A70', 'A72', 'Z16', 'Z17']:
    _rotation_matrix = pd.read_csv(os.path.join(alignment_dir,f'{donor}_alignment_files', f'matrix.csv'), header=None).values

    # 1. cell centroids in microns, as (x, y)
    _xenium_coords = xenium_adata[xenium_adata.obs['DonorID'] == donor].obsm['spatial']#[:, [1, 0]]  # swap cols -> (x, y)
    # 2. microns -> Xenium pixels
    _xenium_coords_px = _xenium_coords / PIXEL_SIZE  # (x, y) in Xenium pixel space

    # 3. Xenium pixels -> H&E pixels (inverse, because the Explorer matrix maps H&E -> Xenium)
    Minv = np.linalg.inv(_rotation_matrix)
    xenium_coords_transformed = np.vstack([xenium_coords_transformed, apply_affine(Minv, _xenium_coords_px)])  # (x, y) in H&E pixel space

    # Save obs_names to add back into main anndata
    xenium_obs_names.extend(xenium_adata[xenium_adata.obs['DonorID'] == donor].obs_names)

# Concatenate and align xenium_coords_transformed to xenium_adata.obs_names order
xenium_coords_transformed = pd.DataFrame(xenium_coords_transformed, index=xenium_obs_names, columns=['x_he_px', 'y_he_px'])
xenium_coords_transformed = xenium_coords_transformed.reindex(xenium_adata.obs_names)

# Quick sanity checks
print(f"Reindexed shape: {xenium_coords_transformed.shape}")
print(f"Missing coords: {xenium_coords_transformed['x_he_px'].isna().sum()} / {xenium_coords_transformed.shape[0]}")

# Add transformed coordinates to xenium_adata.obsm
xenium_adata.obsm['spatial_transformed'] = xenium_coords_transformed.values * scale

# Add H&E sample IDs to xenium_adata.obs for downstream analysis
xenium_adata.obs['SampleID_he'] = xenium_adata.obs['DonorID'].astype(str) + '.ome'

# Fast vectorised assignment of cells to patches based on transformed coordinates
xenium_adata.obs["patch_col"] = (xenium_adata.obsm['spatial_transformed'][:, 0] // patch_px).astype(int)
xenium_adata.obs["patch_row"] = (xenium_adata.obsm['spatial_transformed'][:, 1] // patch_px).astype(int)

# Filter xenium_adata.obs outside tissue mask patches
he_adata_patching = he_adata.obs[['SampleID', 'patch_col', 'patch_row', 'leiden_r0.5', 'leiden_r1.0', 'leiden_r2.0']].reset_index(names='patchID').drop_duplicates().copy()
xenium_adata.obs.drop(columns=he_adata_patching.columns.difference(['SampleID_he', 'patch_col', 'patch_row']), errors='ignore', inplace=True)  # Drop columns if they exist to avoid conflicts
xenium_adata.obs = xenium_adata.obs.reset_index(names='index').merge(he_adata_patching, left_on = ['SampleID_he', 'patch_col', 'patch_row'], right_on = ['SampleID', 'patch_col', 'patch_row'], how = 'left', validate = "many_to_one").set_index('index')

xenium_adata.obs.head()

In [ ]:
# Sanity check: how many cells are outside tissue mask patches?
for lib in ['Z16_FFPE1_01', 'Z17_FFPE1_01', 
            'A72_FFPE1_01', 'A67_FFPE1_01', 
            'A70_FFPE2_01'
            ][:1]:
    
#    ['Capsule/PVS', 'Cortex', 'Medulla', 'mcTEC-rich', 'InflammatoryPVS', 'Adipose']
    sq.pl.spatial_scatter(
        xenium_adata, 
        color='leiden_r0.5', 
        library_key='Sample_hr',
        library_id = lib,
         title = [''],
        legend_loc = None,
        shape=None,
        size=3, 
        img=None,
        palette = 'tab20',
        figsize=(6,6),
        ncols=3, frameon=False, #save=f'{plot_path}/Xenium_cellcharter_{lib}_niche6_thyAge_tinted.png'
        )
    plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
from typing import Union

res = 2.0
leiden_col = 'leiden_r' + str(res)
patch_n = xenium_adata.obs.groupby(leiden_col).size()

min_patches = 1000
valid_cat = patch_n.loc[patch_n >= min_patches].index.tolist()

xenium_adata.obs['spatial_cat'] = xenium_adata.obs[leiden_col].where(xenium_adata.obs[leiden_col].isin(valid_cat), other=pd.NA)
xenium_adata.obs['spatial_cat'] = xenium_adata.obs['spatial_cat'].cat.remove_unused_categories()
xenium_adata.obs['spatial_cat'] = pd.Categorical(xenium_adata.obs['spatial_cat'], categories = sorted(xenium_adata.obs['spatial_cat'].cat.categories, key = lambda x: int(x)))

# Use a single, global color mapping for the clustering
cats = he_adata.obs[leiden_col].cat.categories.tolist()
colors = he_adata.uns[leiden_col + '_colors']
colors = [plt.matplotlib.colors.rgb2hex(c) for c in colors]
palette = dict(zip(cats, colors))

# Manually define colors to obs column where Squidpy expects or creates them
xenium_adata.obs["spatial_cat_colors"] = xenium_adata.obs["spatial_cat"].map(palette)
#del xenium_adata.uns["spatial_cat_colors"] 


for lib in ['Z16_FFPE1_01', 'Z17_FFPE1_01', 'A72_FFPE1_01', 'A67_FFPE1_01', 'A70_FFPE2_01'
            ]:
    
    sq.pl.spatial_scatter(
                xenium_adata,
                library_key='Sample_hr',
                library_id=[lib],
                title=[lib],
                color = 'spatial_cat',
                #palette = ListedColormap(list(palette.values())),
                shape=None,
                size=0.1,
                img=None,
                figsize=calc_figsize(width = 50, height = 50),
                ncols=3,
                frameon=False,
                return_ax=True,
                dpi = 300,
                save = f'{plots_path}/morpho_niches2/thyAging_morphoFM_patch_leiden_m2_r{res}_{lib}_spatial.png',
            )
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

def cluster_enrichment(obs, leiden_col, anno_col, donor_col='DonorID', pseudocount=0.5):
    """
    Per-donor log2 observed/expected enrichment of each cell type in each cluster,
    aggregated across donors with a one-sample t-test (H0: log2 O/E = 0).
    O/E normalises out cluster size AND global cell-type abundance, so it is not
    dominated by abundant populations (e.g. DP thymocytes).
    """
    recs = []
    for donor, g in obs.groupby(donor_col, observed=True):
        ct = pd.crosstab(g[leiden_col], g[anno_col])
        N   = ct.values.sum()
        row = ct.sum(1).values[:, None]     # cells per cluster
        col = ct.sum(0).values[None, :]     # cells per cell type
        expected = row @ col / N
        log2oe = np.log2((ct.values + pseudocount) / (expected + pseudocount))
        tmp = pd.DataFrame(log2oe, index=ct.index, columns=ct.columns)
        tmp = tmp.reset_index().melt(id_vars=leiden_col, var_name=anno_col,
                                     value_name='log2oe')
        tmp[donor_col] = donor
        recs.append(tmp)
    long = pd.concat(recs, ignore_index=True)

    def agg(s):
        s = s.dropna()
        t, p = (stats.ttest_1samp(s, 0.0) if len(s) > 1 and s.std() > 0
                else (np.nan, np.nan))
        return pd.Series({'mean_log2oe': s.mean(), 'std_log2oe': s.std(ddof=1),
                          'n_donors': len(s), 't': t, 'pval': p})

    summary = (long.groupby([leiden_col, anno_col], observed=True)['log2oe']
                   .apply(agg).unstack().reset_index())
    m = summary['pval'].notna()
    summary.loc[m, 'padj'] = multipletests(summary.loc[m, 'pval'], method='fdr_bh')[1]
    return summary, long

patch_summary = xenium_adata.obs[['DonorID', 'Age_group','patchID', 'niche_cellcharter6', 'taa_l3', 'taa_l2_v2', 'leiden_r0.5', 'leiden_r1.0', 'leiden_r2.0']].copy()
summary, long = cluster_enrichment(patch_summary, 'leiden_r2.0', 'taa_l3')


In [ ]:
plot_df = summary.pivot(index='leiden_r2.0', columns='taa_l3', values='mean_log2oe')
plot_df.fillna(0, inplace=True)

# Order axes if categorical
if hasattr(summary['leiden_r2.0'], 'cat'):
    plot_df = plot_df.reindex(index=summary['leiden_r2.0'].cat.categories)
if hasattr(summary['taa_l3'], 'cat'):
    plot_df = plot_df.reindex(columns=summary['taa_l3'].cat.categories)

from plotting.utils import create_diverging_palette
cmap = create_diverging_palette(thyAgeing_colors['teal'], thyAgeing_colors['orange'], s = 90, l = 60, as_cmap = True)

vmax = 3
fig, ax = plt.subplots(figsize=calc_figsize(width=220, height=160))
g = sns.clustermap(
    plot_df,
    cmap=cmap,
    center=0,
    vmin=-vmax,
    vmax=vmax,
    figsize=calc_figsize(width=220, height=160),
    cbar_kws={'label': 'mean_log2oe'},
    row_cluster=True,
    col_cluster=True,
    #dendrogram_ratio=(0.06, 0.06),   # reduce dendrogram -> heatmap gap
    cbar_pos=(0.02, 0.75, 0.03, 0.18),
    tree_kws={'linewidth': 0.5},
    xticklabels=True,
    yticklabels=True,
)
ax = g.ax_heatmap
plt.setp(ax.get_xticklabels(), fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
ax.set_xlabel('taa_l3')
ax.set_ylabel('leiden_r2.0')
#plt.tight_layout()
plt.savefig(f'{plots_path}/morpho_niches2/thyAging_morphoFM_patch_leiden_r2.0_taa_l3_log2oe_clustermap.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
plot_df.iloc[g.dendrogram_row.reordered_ind].reset_index(names='leiden_r2.0')['leiden_r2.0'].to_csv(f'/nfs/team205/lm25/thymus_projects/thymus_ageing_atlas/Spatial_analyses/data/morpho_niches/thyAging_morphoFM_patch_leiden_r2.0_taa_l3_log2oe_clustermap_row_order.csv', index=False)

In [ ]:
# Plot palette swatches
palette2 = dict(zip(xenium_adata.obs['spatial_cat'].cat.categories, xenium_adata.uns['spatial_cat_colors']))
colors = list(palette2.items())
n = len(colors)
fig, ax = plt.subplots(figsize=(n * 0.4, 2))
ax.bar(range(n), [1]*n, color=[c for _, c in colors], edgecolor='none')
ax.set_xticks(range(n))
ax.set_xticklabels([k for k, _ in colors], rotation=45, ha='right', fontsize=8)
ax.set_yticks([])
ax.set_xlim(-0.5, n-0.5)
ax.set_title('palette')
plt.tight_layout()
plt.show()

niche 10 maybe mcTEC-rich

In [ ]:
import seaborn as sns

# Percentual niche composition per DonorID for leiden_m2_r{res}
res = 1.0
ct = pd.crosstab(xenium_adata.obs['DonorID'], xenium_adata.obs[f'leiden_r{res}'])
pct = ct.div(ct.sum(axis=1), axis=0) * 100

# Consistent numeric column order
cols = sorted(pct.columns.astype(str), key=lambda x: int(x) if x.isdigit() else x)
pct = pct.reindex(columns=cols, fill_value=0)

# Order rows by donor age
donor_order = xenium_adata.obs.drop_duplicates('DonorID').set_index('DonorID').loc[pct.index].sort_values('age_months')['age_months'].index
pct = pct.reindex(index=donor_order)

# Exclude small clusters
pct = pct.loc[:, valid_cat]
fig, ax = plt.subplots(figsize=calc_figsize(width=100, height=100))

x = np.arange(len(pct.index))
bottom = np.zeros(len(pct.index), dtype=float)

for c in pct.columns:
    values = pct[c].to_numpy()
    ax.bar(
        x,
        values,
        bottom=bottom,
        color=palette2.get(str(c), "#cccccc"),
        width=0.8,
        label=str(c),
        edgecolor="none",
    )
    bottom += values

ax.set_xticks(x)
ax.set_xticklabels(pct.index)
ax.set_xlabel("DonorID")
ax.set_ylabel("Percent")
ax.set_title(f"Percentual composition of leiden_r{res} per DonorID")
ax.legend(title=f"leiden_r{res}", bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine(ax=ax, trim=True, offset=(0, 5))

plt.tight_layout()
plt.savefig(f'{plots_path}/morpho_niches2/thyAging_morphoFM_patch_mf2_leiden_m2_r{res}_niche_composition_per_DonorID.png', dpi=300)
plt.show()

In [ ]:
patch_summary = xenium_adata.obs[['DonorID', 'Age_group','patchID', 'niche_cellcharter6', 'taa_l3', 'taa_l2_v2', 'leiden_r0.5', 'leiden_r1.0', 'leiden_r2.0']].copy()

res = 2.0
leiden_col = f'leiden_r{res}'
anno_col = 'taa_l3'
df = patch_summary.groupby(['DonorID', leiden_col, anno_col], observed = True).size().to_frame(name='count').reset_index()
df = df.pivot_table(index=['DonorID', leiden_col], columns=anno_col, values='count', fill_value=0)
df_colsums = df.values.sum(axis = 0)
df = df/df_colsums

df_means = df.reset_index().melt(id_vars=['DonorID', leiden_col]).groupby([leiden_col, anno_col], observed=True).agg(mean = ('value', 'mean'), std = ('value', 'std'))
df_means['std_pos'] = df_means['mean'] + df_means['std']
df_means['mean'] = df_means['mean']
df_means = df_means.reset_index()
df_means

In [ ]:
import seaborn as sns
# Heatmap-like plot with size and color mapped to mean
plot_df = df_means.copy()
plot_df['mean'] = plot_df['mean'].clip(upper=0.5)  # Avoid zero values for better visualization
plot_df[leiden_col] = plot_df[leiden_col].cat.remove_unused_categories()
plot_df[leiden_col] = pd.Categorical(plot_df[leiden_col], categories=sorted(plot_df[leiden_col].unique(), key=lambda x: int(x)))

x_order = plot_df[leiden_col].cat.categories.tolist() if hasattr(plot_df[leiden_col], 'cat') else sorted(plot_df[leiden_col].unique(), key=lambda x: int(x))
y_order = plot_df[anno_col].unique().tolist()

fig, ax = plt.subplots(figsize=calc_figsize(width=len(x_order)*3+20, height=200))

sns.scatterplot(
    data=plot_df,
    x=leiden_col,
    y=anno_col,
    hue='mean',
    size='mean',
    sizes=(0, 400),
    palette=sns.blend_palette(
        [
            thyAgeing_colors['blue'],
            thyAgeing_colors['purple'],
            thyAgeing_colors['magenta'],
            thyAgeing_colors['orange'],
            thyAgeing_colors['yellow'],
        ],
        as_cmap=True,
        n_colors=10,
    ),
    hue_norm=(plot_df['mean'].min(), plot_df['mean'].max()),
    size_norm=(plot_df['mean'].min(), plot_df['mean'].max()),
    marker='.',
    linewidth=0,
    edgecolor=None,
    ax=ax,
    legend='brief',
    clip_on=False,
)

ax.set_xlabel(leiden_col)
ax.set_ylabel(anno_col)
ax.set_xticks(range(len(x_order)))
ax.set_xticklabels(x_order, rotation=45, ha='right')
ax.set_yticks(range(len(y_order)))
ax.set_yticklabels(y_order)
ax.legend(title='Mean proportion', bbox_to_anchor=(1.02, 1), loc='upper left')

# Add padding so largest dots are not clipped at plot borders
ax.set_xlim(-0.7, len(x_order) - 0.3)
ax.set_ylim(-0.7, len(y_order) - 0.3)

sns.despine(ax=ax, offset = (3,1), trim=True)
plt.tight_layout()
plt.savefig(f'{plots_path}/morpho_niches2/thyAging_morphoFM_patch_mf2_{leiden_col}_{anno_col}_dotplot.pdf', dpi=300)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

def cluster_enrichment(obs, leiden_col, anno_col, donor_col='DonorID', pseudocount=0.5):
    """
    Per-donor log2 observed/expected enrichment of each cell type in each cluster,
    aggregated across donors with a one-sample t-test (H0: log2 O/E = 0).
    O/E normalises out cluster size AND global cell-type abundance, so it is not
    dominated by abundant populations (e.g. DP thymocytes).
    """
    recs = []
    for donor, g in obs.groupby(donor_col, observed=True):
        ct = pd.crosstab(g[leiden_col], g[anno_col])
        N   = ct.values.sum()
        row = ct.sum(1).values[:, None]     # cells per cluster
        col = ct.sum(0).values[None, :]     # cells per cell type
        expected = row @ col / N
        log2oe = np.log2((ct.values + pseudocount) / (expected + pseudocount))
        tmp = pd.DataFrame(log2oe, index=ct.index, columns=ct.columns)
        tmp = tmp.reset_index().melt(id_vars=leiden_col, var_name=anno_col,
                                     value_name='log2oe')
        tmp[donor_col] = donor
        recs.append(tmp)
    long = pd.concat(recs, ignore_index=True)

    def agg(s):
        s = s.dropna()
        t, p = (stats.ttest_1samp(s, 0.0) if len(s) > 1 and s.std() > 0
                else (np.nan, np.nan))
        return pd.Series({'mean_log2oe': s.mean(), 'std_log2oe': s.std(ddof=1),
                          'n_donors': len(s), 't': t, 'pval': p})

    summary = (long.groupby([leiden_col, anno_col], observed=True)['log2oe']
                   .apply(agg).unstack().reset_index())
    m = summary['pval'].notna()
    summary.loc[m, 'padj'] = multipletests(summary.loc[m, 'pval'], method='fdr_bh')[1]
    return summary, long

summary, long = cluster_enrichment(patch_summary, 'leiden_r2.0', 'taa_l3')
